In [1]:
%load_ext watermark
%watermark -a "Manuel Elias Orellana Lavayen" -d -v -iv

Author: Manuel Elias Orellana Lavayen

Date: 2026-08-29

Python implementation: CPython
Python version       : 3.12.10
IPython version      : 9.17.0



# Librerias

In [2]:
%load_ext cuml.accel
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
from sentence_transformers import SentenceTransformer
from pathlib import Path
ruta_base = Path.cwd() #Ruta base
import skops.io as sio
import numpy as np
from scipy.sparse import save_npz

In [3]:
ruta_base

PosixPath('/espacio_trabajo/Analisis de datos y entrenamiento de modelos')

# Cargando Datasets Limpios

In [4]:
# Cargar datasets limpios
df_train = pd.read_csv(filepath_or_buffer=ruta_base / "Datasets Limpios y corregidos/dataset_train.csv")
df_validation = pd.read_csv("./Datasets Limpios y corregidos/dataset_validation.csv")
df_test = pd.read_csv("./Datasets Limpios y corregidos/dataset_test.csv")

print("Train:", df_train.shape)
print("Validation:", df_validation.shape)
print("Test:", df_test.shape)

print("\nNulos:")
print("Train:", df_train.isna().sum().sum())
print("Validation:", df_validation.isna().sum().sum())
print("Test:", df_test.isna().sum().sum())

print("\nDuplicados:")
print("Train:", df_train.duplicated().sum())
print("Validation:", df_validation.duplicated().sum())
print("Test:", df_test.duplicated().sum())

Train: (198432, 7)
Validation: (4994, 7)
Test: (4992, 7)

Nulos:
Train: 0
Validation: 0
Test: 0

Duplicados:
Train: 0
Validation: 0
Test: 0


# Separación de datos

In [5]:
# Textos para TF-IDF
X_train_tfidf_texto = df_train["text_tfidf"]
X_validation_tfidf_texto = df_validation["text_tfidf"]
X_test_tfidf_texto = df_test["text_tfidf"]

# Textos para Embeddings
X_train_embeddings_texto = df_train["text_embedding"]
X_validation_embeddings_texto = df_validation["text_embedding"]
X_test_embeddings_texto= df_test["text_embedding"]

# Textos para TF-IDF con  Correcion Ortografica
X_train_tfidf_texto_ortografia = df_train["text_tfidf_ortografia"]
X_validation_tfidf_texto_ortografia  = df_validation["text_tfidf_ortografia"]
X_test_tfidf_texto_ortografia  = df_test["text_tfidf_ortografia"]

# Textos para Embeddings con Correcion Ortografica
X_train_embeddings_texto_ortografia  = df_train["text_embedding_ortografia"]
X_validation_embeddings_texto_ortografia  = df_validation["text_embedding_ortografia"]
X_test_embeddings_texto_ortografia = df_test["text_embedding_ortografia"]

# Etiquetas
y_train = df_train["label"]
y_validation = df_validation["label"]
y_test = df_test["label"]

# Vectorización

### Funciones de vectorizacion

In [6]:
# Vectorización Embeddings
ruta_vectorizador= ruta_base/"Recursos de procesamiento de texto/multilingual-e5-base"
modelo_vectorizador = SentenceTransformer(str(ruta_vectorizador))

def vectorizar_embeddings(textos):
    """
    Vectoriza con embeddings los textos de una Serie de Pandas utilizando el modelo multilingual-e5-base

    Args:
        textos: Serie de Pandas

    Returns:
        vectores embeddings
    """

    textos_prefijo = []

    for texto in textos:
        textos_prefijo.append(f"passage: {texto}") #El prefijo es necesario para que el modelo funcione

    textos = textos_prefijo

    embeddings = modelo_vectorizador.encode(
        textos,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    return embeddings

# Vectorización TF-IDF
def vectorizar_tfidf(textos, max_features, n_grama, sublinear_tf, vectorizador = None):
    """
    Vectoriza con TF-IDF los textos de una Serie de Pandas utilizando el vectorizador TF-IDF de Sklearn

    Args:
        textos: Serie de Pandas
        max_features: Maximo de caracteristicas
        n_grama : Cantidad de ngramas de (1, n)
        sublinear_tf: Aplicar escalado logaritmico
        vectorizador: Vectorizador a usar (En caso de no mandarse, crea y entrena uno nuevo)

    Returns:
        vectorizador
        matriz (vectores TF-IDF)
    """
    if vectorizador is None:
        vectorizador = TfidfVectorizer(
            max_features=max_features, #Número maximo de caracteristicas (palabras) entre unigramas y trigramas
            ngram_range=(1,n_grama),#Unigramas y Bigramas
            min_df=15, #Borra palabras que no aparecen en al menos 10 docs
            max_df=0.70, #Borra palabras que aparecen en el 70% de los docs (Muy comunes),
            sublinear_tf=sublinear_tf
        )
        matriz = vectorizador.fit_transform(textos)
    else:
        matriz = vectorizador.transform(textos)

    return vectorizador, matriz


def ejecutar_vectorizacion_tfidf(
    X_train_texto,
    X_validation_texto,
    X_test_texto,
    max_features_list=(10000, 20000, 30000, 40000, 50000),
    n_grama_list=(2, 3, 4),
    sublinear_tf_list=(True, False),
):
    """
    Aplica la funcion de vectorizar_tfidf con diferentes textos y diferentes parametros

    Args:
        X_train_texto: Serie de pandas con textos de entrenamiento
        X_validation_texto: Serie de pandas con textos de validación
        X_test_texto: Serie de pandas con textos de testeo
        max_features_list: lista de Maximo de caracteristicas
        n_grama_list : lista de Cantidad de ngramas de (1, n)
        sublinear_tf_list: lista booleana de escalado logaritmico

    Returns:
        Diccionario con todas las combinaciones de parametros
    """

    resultados = {}

    total_combinaciones = (len(max_features_list)* len(n_grama_list)* len(sublinear_tf_list))
    combinacion_actual = 0

    print("=" * 70)
    print("INICIANDO VECTORIZACIÓN TF-IDF")
    print(f"Total de combinaciones: {total_combinaciones}")
    print("=" * 70)

    for sublinear_tf in sublinear_tf_list:
        clave_sublinear = f"Sublineal_{sublinear_tf}"
        resultados[clave_sublinear] = {}

        for n_grama in n_grama_list:
            clave_ngram = f"ngram_{n_grama}"
            resultados[clave_sublinear][clave_ngram] = {}

            for max_features in max_features_list:
                combinacion_actual += 1
                clave_features = f"max_features_{max_features}"

                print("\n" + "-" * 70)
                print(f"COMBINACIÓN {combinacion_actual}/{total_combinaciones}")
                print(f"  Sublinear TF : {sublinear_tf}")
                print(f"  N-gram       : 1-{n_grama}")
                print(f"  Max features : {max_features}")
                print("-" * 70)

                print("  [1/3] Vectorizando TRAIN...")
                vectorizador, X_train_vectorizado = vectorizar_tfidf(
                    textos=X_train_texto,
                    max_features=max_features,
                    n_grama=n_grama,
                    sublinear_tf=sublinear_tf,
                    vectorizador=None
                )

                print("  [2/3] Vectorizando VALIDATION...")
                _, X_validation_vectorizado = vectorizar_tfidf(
                    textos=X_validation_texto,
                    max_features=max_features,
                    n_grama=n_grama,
                    sublinear_tf=sublinear_tf,
                    vectorizador=vectorizador
                )

                print("  [3/3] Vectorizando TEST...")
                _, X_test_vectorizado = vectorizar_tfidf(
                    textos=X_test_texto,
                    max_features=max_features,
                    n_grama=n_grama,
                    sublinear_tf=sublinear_tf,
                    vectorizador=vectorizador
                )

                # GUARDAR EN DICCIONARIO

                resultados[clave_sublinear][clave_ngram][clave_features] = {
                    "vectorizador": vectorizador,
                    "X_train": X_train_vectorizado,
                    "X_validation": X_validation_vectorizado,
                    "X_test": X_test_vectorizado,
                }

                print(f"COMBINACIÓN {combinacion_actual} TERMINADA")

    print("TODAS LAS VECTORIZACIONES HAN TERMINADO")

    return resultados

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Aplicando Funciones de vectorización

#### TF - IDF

In [7]:
#Vectorizando TF_IDF No Ortografia
vectorizacion_tfidf_normal= ejecutar_vectorizacion_tfidf(X_train_tfidf_texto, X_validation_tfidf_texto, X_test_tfidf_texto)

INICIANDO VECTORIZACIÓN TF-IDF
Total de combinaciones: 30

----------------------------------------------------------------------
COMBINACIÓN 1/30
  Sublinear TF : True
  N-gram       : 1-2
  Max features : 10000
----------------------------------------------------------------------
  [1/3] Vectorizando TRAIN...
  [2/3] Vectorizando VALIDATION...
  [3/3] Vectorizando TEST...
COMBINACIÓN 1 TERMINADA

----------------------------------------------------------------------
COMBINACIÓN 2/30
  Sublinear TF : True
  N-gram       : 1-2
  Max features : 20000
----------------------------------------------------------------------
  [1/3] Vectorizando TRAIN...
  [2/3] Vectorizando VALIDATION...
  [3/3] Vectorizando TEST...
COMBINACIÓN 2 TERMINADA

----------------------------------------------------------------------
COMBINACIÓN 3/30
  Sublinear TF : True
  N-gram       : 1-2
  Max features : 30000
----------------------------------------------------------------------
  [1/3] Vectorizando TRAIN..

In [8]:
#Vectorizando TF_IDF Si Ortografia
vectorizacion_tfidf_ortografia= ejecutar_vectorizacion_tfidf(X_train_tfidf_texto_ortografia, X_validation_tfidf_texto_ortografia, X_test_tfidf_texto_ortografia)

INICIANDO VECTORIZACIÓN TF-IDF
Total de combinaciones: 30

----------------------------------------------------------------------
COMBINACIÓN 1/30
  Sublinear TF : True
  N-gram       : 1-2
  Max features : 10000
----------------------------------------------------------------------
  [1/3] Vectorizando TRAIN...
  [2/3] Vectorizando VALIDATION...
  [3/3] Vectorizando TEST...
COMBINACIÓN 1 TERMINADA

----------------------------------------------------------------------
COMBINACIÓN 2/30
  Sublinear TF : True
  N-gram       : 1-2
  Max features : 20000
----------------------------------------------------------------------
  [1/3] Vectorizando TRAIN...
  [2/3] Vectorizando VALIDATION...
  [3/3] Vectorizando TEST...
COMBINACIÓN 2 TERMINADA

----------------------------------------------------------------------
COMBINACIÓN 3/30
  Sublinear TF : True
  N-gram       : 1-2
  Max features : 30000
----------------------------------------------------------------------
  [1/3] Vectorizando TRAIN..

#### Embeddings

In [9]:
X_train_embeddings_vectorizados_normal= vectorizar_embeddings(X_train_embeddings_texto)
X_validation_embeddings_vectorizados_normal= vectorizar_embeddings(X_validation_embeddings_texto)
X_test_embeddings_vectorizados_normal= vectorizar_embeddings(X_test_embeddings_texto)

X_train_embeddings_vectorizados_ortografia= vectorizar_embeddings(X_train_embeddings_texto_ortografia)
X_validation_embeddings_vectorizados_ortografia= vectorizar_embeddings(X_validation_embeddings_texto_ortografia)
X_test_embeddings_vectorizados_ortografia= vectorizar_embeddings(X_test_embeddings_texto_ortografia)

Batches:   0%|          | 0/6201 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Batches:   0%|          | 0/156 [00:00<?, ?it/s]

Batches:   0%|          | 0/6201 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Batches:   0%|          | 0/156 [00:00<?, ?it/s]

# Guardando Y, Matrices y Embeddings

In [10]:
#Guardando embeddings

#Normales
np.save("./matrices y embeddings/No Ortografia/Embeddings/X_train_embeddings.npy", X_train_embeddings_vectorizados_normal)
np.save("./matrices y embeddings/No Ortografia/Embeddings/X_validation_embeddings.npy", X_validation_embeddings_vectorizados_normal)
np.save("./matrices y embeddings/No Ortografia/Embeddings/X_test_embeddings.npy", X_test_embeddings_vectorizados_normal)

#Ortografia
np.save("./matrices y embeddings/Ortografia/Embeddings/X_train_embeddings.npy", X_train_embeddings_vectorizados_ortografia)
np.save("./matrices y embeddings/Ortografia/Embeddings/X_validation_embeddings.npy", X_validation_embeddings_vectorizados_ortografia)
np.save("./matrices y embeddings/Ortografia/Embeddings/X_test_embeddings.npy", X_test_embeddings_vectorizados_ortografia)

#Guardar Y
np.save("./Y/y_train.npy", y_train.to_numpy())
np.save("./Y/y_validation.npy", y_validation.to_numpy())
np.save("./Y/y_test.npy", y_test.to_numpy())

### Función Para Guardado de Matrices TF-IDF (Se aplicó esta funcion ya que son varias matrices)

In [17]:
def guardar_vectorizaciones_tfidf(
    vectorizaciones,
    ruta_base
):
    """
    Guarda en carpetas (Ya creadas anteriormente), los vectores TF-IDF del del diccionario

    Args:
        vectorizaciones: Didccionarios de vectores TF-IDF
        ruta_base: Ruta base, la puedes obtener con -> from pathlib import Path -> Path.cwd()

    """

    ruta_base = Path(ruta_base)

    for sublinear, datos_sublinear in vectorizaciones.items():
        for ngram, datos_ngram in datos_sublinear.items():
            for max_features, datos in datos_ngram.items():
                # Extraer valores de las claves
                sublinear_valor = sublinear.replace("Sublineal_","").lower()
                ngram_valor = ngram.replace("ngram_","")
                max_features_valor = max_features.replace("max_features_","")

                # Crear ruta
                ruta = (ruta_base/ f"sublinear_{sublinear_valor}"/ f"ngram_{ngram_valor}"/ max_features_valor)
                ruta.mkdir(parents=True,exist_ok=True)

                # Guardar matrices
                save_npz(ruta / "X_train_tfidf.npz",datos["X_train"])
                save_npz(ruta / "X_validation_tfidf.npz",datos["X_validation"])
                save_npz(ruta / "X_test_tfidf.npz",datos["X_test"])

                #Guardar vectorizador
                sio.dump(datos["vectorizador"], ruta/"vectorizador_tfidf.skops")

                print(
                    f"Guardado: "
                    f"sublinear_{sublinear_valor} | "
                    f"ngram_{ngram_valor} | "
                    f"max_features={max_features_valor}"
                )

In [18]:
guardar_vectorizaciones_tfidf(vectorizacion_tfidf_normal,"./matrices y embeddings/No Ortografia/Matrices TF-IDF")
guardar_vectorizaciones_tfidf(vectorizacion_tfidf_ortografia,"./matrices y embeddings/Ortografia/Matrices TF-IDF")

Guardado: sublinear_true | ngram_2 | max_features=10000
Guardado: sublinear_true | ngram_2 | max_features=20000
Guardado: sublinear_true | ngram_2 | max_features=30000
Guardado: sublinear_true | ngram_2 | max_features=40000
Guardado: sublinear_true | ngram_2 | max_features=50000
Guardado: sublinear_true | ngram_3 | max_features=10000
Guardado: sublinear_true | ngram_3 | max_features=20000
Guardado: sublinear_true | ngram_3 | max_features=30000
Guardado: sublinear_true | ngram_3 | max_features=40000
Guardado: sublinear_true | ngram_3 | max_features=50000
Guardado: sublinear_true | ngram_4 | max_features=10000
Guardado: sublinear_true | ngram_4 | max_features=20000
Guardado: sublinear_true | ngram_4 | max_features=30000
Guardado: sublinear_true | ngram_4 | max_features=40000
Guardado: sublinear_true | ngram_4 | max_features=50000
Guardado: sublinear_false | ngram_2 | max_features=10000
Guardado: sublinear_false | ngram_2 | max_features=20000
Guardado: sublinear_false | ngram_2 | max_feat